# Capstone — Modern Data Engineering for AI Systems

**Team:** Alanoud Alotaibi, Rawan H., Reem Alshathri | SDAIA Academy | Trainer: Mohammed Albeladi

## What this is

This notebook runs the full capstone pipeline end-to-end on the
[Customer Support Tickets - CRM dataset](https://www.kaggle.com/datasets/ajverse/customer-support-tickets-crm-dataset).

Raw CRM ticket exports are messy: inconsistent priority/status values, blank descriptions,
missing satisfaction scores. Loading that straight into an analytics table gives wrong numbers
with no way to tell. So this pipeline streams every row through Kafka and checks it against a
Pydantic contract; anything that fails is quarantined with a reason instead of disappearing.
What passes lands in a Bronze/Silver/Gold Delta Lakehouse, with a real `MERGE` upsert into
Silver and a Great Expectations gate that has to pass before Gold gets built. On top of the
clean data sits a hybrid-search RAG system over historical resolutions, so a support agent can
ask "how was this handled before?" and get an answer grounded in, and cited to, real past
tickets.

We saved this notebook **with its output**, because the output is the evidence the pipeline
actually ran — not just that the code compiles.

## The five parts

1. **Ingestion** — real `kafka-python` producer/consumer, Pydantic data contract, quarantine + DLQ
2. **Delta Lakehouse** — Bronze / Silver / Gold, real `MERGE` upsert, schema enforcement
3. **RAG** — chunking, ChromaDB, BM25, RRF fusion, cross-encoder reranking, citations
4. **Orchestration** — the Airflow DAG that wires all of it together, and halts on a bad gate
5. **Quality gate + lineage** — Great Expectations checkpoint, OpenLineage START/COMPLETE/FAIL



## Step 0 — Get the repo onto this Colab machine

Run **one** of the two cells below, not both.

In [ ]:
# Option A — repo is already on GitHub (recommended)
import os

REPO_DIR = "/content/sdaia-capstone-data-pipeline"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Alanoud-Alotaibi/sdaia-capstone-data-pipeline.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}


In [ ]:
# Option B — you haven't pushed to GitHub yet: upload the project zip instead
# 1. Run this cell
# 2. Click "Choose Files" and pick sdaia-capstone-data-pipeline.zip
# 3. It unzips to /content/sdaia-capstone-data-pipeline
from google.colab import files
import zipfile, os

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")

REPO_DIR = "/content/repo"  # the zip contains a top-level "repo" folder
%cd {REPO_DIR}


## Step 1 — Install dependencies

In [ ]:
!pip install -q pyspark==3.5.0 delta-spark==3.2.0 kafka-python==2.0.2 "pydantic>=2.0" \
    kagglehub chromadb sentence-transformers rank-bm25 "apache-airflow>=2.8" \
    "great-expectations>=1.0" openlineage-python


## Step 2 — Get the dataset from Kaggle

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub
dataset_path = kagglehub.dataset_download("ajverse/customer-support-tickets-crm-dataset")
print(dataset_path)
os.listdir(dataset_path)


In [ ]:
import pandas as pd

# Find the CSV(s) in the downloaded dataset and take a look at the real columns.
csv_files = [f for f in os.listdir(dataset_path) if f.endswith(".csv")]
print("CSV files:", csv_files)

preview = pd.read_csv(os.path.join(dataset_path, csv_files[0]))
print(preview.columns.tolist())
preview.head()


**Check the printed column list against `src/config.py::COLUMN_MAP` before continuing.**
If a name doesn't match (e.g. Kaggle uses `Ticket Priority` but this shows something else),
edit `COLUMN_MAP` now — every downstream module reads through it, so it's the only place that
needs to change.

In [ ]:
import shutil, os

os.makedirs("/content/capstone-data/raw", exist_ok=True)
target_csv = "/content/capstone-data/raw/tickets.csv"
shutil.copy(os.path.join(dataset_path, csv_files[0]), target_csv)
target_csv


## Step 3 — Start a local Kafka broker

In [ ]:
%%bash
if [ ! -d "kafka_2.13-3.7.0" ]; then
  curl -sSL https://downloads.apache.org/kafka/3.7.0/kafka_2.13-3.7.0.tgz -o kafka.tgz
  tar -xzf kafka.tgz
fi
echo "Kafka binaries ready."

In [ ]:
%%bash
cd kafka_2.13-3.7.0
nohup bin/zookeeper-server-start.sh config/zookeeper.properties > /content/zookeeper.log 2>&1 &
sleep 8
nohup bin/kafka-server-start.sh config/server.properties > /content/kafka.log 2>&1 &
sleep 8
echo "Kafka + Zookeeper started."

In [ ]:
import socket, time

def port_open(host, port, timeout=1):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

for _ in range(15):
    if port_open("localhost", 9092):
        print("Kafka is up on 9092")
        break
    time.sleep(2)
else:
    raise RuntimeError("Kafka never came up — check /content/kafka.log")

In [ ]:
%%bash
cd kafka_2.13-3.7.0
bin/kafka-topics.sh --create --if-not-exists --topic support-tickets-raw   --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1
bin/kafka-topics.sh --create --if-not-exists --topic support-tickets-valid --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1
bin/kafka-topics.sh --create --if-not-exists --topic support-tickets-dlq   --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1
bin/kafka-topics.sh --list --bootstrap-server localhost:9092


## Step 4 — Put the project package on the Python path

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)  # REPO_DIR was set in Step 0
print("Using project root:", REPO_DIR)


## Step 5 — Push access (optional, only needed if you're pushing from Colab)

In [ ]:
from google.colab import userdata

github_token = userdata.get("GITHUB")
username = "Alanoud-Alotaibi"
repo = "sdaia-capstone-data-pipeline"

# Run this once per session, then commit as whichever teammate is actually
# authoring that commit — swap the two lines below to match.
!git -C {REPO_DIR} remote set-url origin https://{github_token}@github.com/{username}/{repo}.git
!git -C {REPO_DIR} config user.name "Alanoud-Alotaibi"
!git -C {REPO_DIR} config user.email "<alanoud's github-registered email>"

!git -C {REPO_DIR} config user.name "Rawan1H"
!git -C {REPO_DIR} config user.email "rawan1hamad@hotmail.com"

!git -C {REPO_DIR} config user.name "ReemAlshathri74"
!git -C {REPO_DIR} config user.email "reem.74sh@gmail.com"